# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` — folds, track preparation, metrics, the loop itself
- `src/models/` — model functions and `MODEL_REGISTRY`
- `src/features/quantum/` — quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: /content/capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = "Manuel"  # your name, recorded on every result row (default: OS user)
RESUME        = False # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce"]
ACTIVE_TRACKS = ['A']                         # any keys of TRACK_CONFIG
ACTIVE_MODELS = ['decision_tree']       # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['A']
Models:   ['decision_tree']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track A        target INDPRO     loaded   shape=(767, 123)
  Track A        target PAYEMS     loaded   shape=(767, 123)
  Track A        target CPIAUCSL   loaded   shape=(767, 123)
  Track A        target S&P 500    loaded   shape=(767, 123)
Stationary panel (767, 123)  1962-04-01 -> 2026-02-01
Targets          (806, 20)


In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Saved: raw/decision_tree/A.parquet   new=300  total=300


  Saved: raw/decision_tree/A.parquet   new=300  total=600


  Saved: raw/decision_tree/A.parquet   new=300  total=900


  1000/7764 runs complete · 29s elapsed · ~3m16s left


  Saved: raw/decision_tree/A.parquet   new=300  total=1200


...

  Saved: raw/decision_tree/A.parquet   new=300  total=6900


  7000/7764 runs complete · 3m19s elapsed · ~22s left


  Saved: raw/decision_tree/A.parquet   new=300  total=7200


  Saved: raw/decision_tree/A.parquet   new=300  total=7500


  Saved: raw/decision_tree/A.parquet   new=224  total=7724
Loop complete. Total rows: 7724 in 3m40s
Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae      rmse  \
model         track target   horizon                                            
decision_tree A     CPIAUCSL 1        0.000010  0.002321   0.002321  0.002321   
                             3        0.000046  0.004783   0.004783  0.004783   
                             6        0.000112  0.007280   0.007280  0.007280   
                    INDPRO   1        0.000125  0.006674   0.006674  0.006674   
                             3        0.000487  0.012790   0.012790  0.012790   
                             6        0.000954  0.019564   0.019564  0.019564   
                    PAYEMS   1        0.000052  0.001947   0.001947  0.001947   
                             3        0.000237  0.004756   0.004756  0.004756   
                             6        0.000215  0.005929   0.005929  0.005929   
                    S&P 500  1        0.002051  0.032931   0.032931  0.032931   
                             3        0.006761  0.059768   0.059768  0.059768   
                             6        0.013276  0.082392   0.082392  0.082392   

                                          mase      rmspe  
model         track target   horizon                       
decision_tree A     CPIAUCSL 1        1.220319   2.082161  
                             3        1.009698   2.365195  
                             6        1.048517   1.032510  
                    INDPRO   1        1.041609   5.596908  
                             3        1.036168   7.522965  
                             6        0.947178  18.796311  
                    PAYEMS   1        1.324733   2.659498  
                             3        1.355123   1.327470  
                             6        1.055490   2.125532  
                    S&P 500  1        0.996703   6.635818  
                             3        0.854783   2.958824  
                             6        0.814415   3.074526

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  median_ae  \
model    track target   horizon usrec                                  
adaboost A     CPIAUCSL 1       0      0.000015  0.002551   0.002551   
                                1      0.000019  0.004009   0.004009   
                        3       0      0.000042  0.005282   0.005282   
                                1      0.000015  0.002993   0.002993   
                        6       0      0.000160  0.010710   0.010710   
...                                         ...       ...        ...   
xgboost  B     S&P 500  1       1      0.002684  0.039262   0.039262   
                        3       0      0.004355  0.048121   0.048121   
                                1      0.009794  0.075201   0.075201   
                        6       0      0.007973  0.069094   0.069094   
                                1      0.023473  0.132138   0.132138   

                                           rmse      mase     rmspe  
model    track target   horizon usrec                                
adaboost A     CPIAUCSL 1       0      0.002551  1.599220  0.310822  
                                1      0.004009  2.167473  0.397093  
                        3       0      0.005282  1.564546  0.267708  
                                1      0.002993  0.821885  0.102919  
                        6       0      0.010710  2.283522  0.259396  
...                                         ...       ...       ...  
xgboost  B     S&P 500  1       1      0.039262  1.110735  1.552633  
                        3       0      0.048121  0.688178  1.773647  
                                1      0.075201  1.033627  1.256534  
                        6       0      0.069094  0.633340  2.645870  
                                1      0.132138  1.162902  1.300411  

[192 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,decision_tree,A,CPIAUCSL,1,0,1972-04-01,0.002407,0.003775
1,decision_tree,A,CPIAUCSL,1,1,1972-05-01,0.002401,0.002481
2,decision_tree,A,CPIAUCSL,1,2,1972-06-01,0.002395,0.001785
3,decision_tree,A,CPIAUCSL,1,3,1972-07-01,0.002389,0.005436
4,decision_tree,A,CPIAUCSL,1,4,1972-08-01,0.004762,0.002437


,,mean,sum
model,track,,
decision_tree,A,0.023,175.838
